<a href="https://colab.research.google.com/github/Darklord007masai/part4-fastapi-service/blob/main/tests_test_api_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pytest
from fastapi.testclient import TestClient
import pickle
import numpy as np
import os
from app.main import app, MODEL_PATH

# 1. Test Setup: Generate a mock model file to ensure test isolation and stability
@pytest.fixture(scope="session", autouse=True)
def setup_mock_model_weights():
    """
    Creates a temporary mock model artifact if model.pkl doesn't exist.
    This guarantees the test suite runs successfully even in a clean CI/CD environment.
    """
    if not os.path.exists(MODEL_PATH):
        class MockModel:
            def predict_proba(self, X):
                # Returns a mock probability matrix: 80% non-churn, 20% churn
                return np.array([[0.80, 0.20]] * len(X))

        mock_payload = {'model': MockModel(), 'scaler': None}
        with open(MODEL_PATH, "wb") as f:
            pickle.dump(mock_payload, f)
    yield

# Initialize the FastAPI TestClient
client = TestClient(app)

# --- REQUIRED TEST CASES ---

def test_health_endpoint():
    """
    Test Case 1: Verify the /health endpoint returns a 200 OK status
    and confirms the model is successfully loaded into memory.
    """
    response = client.get("/health")
    assert response.status_code == 200
    assert response.json()["status"] == "healthy"
    assert response.json()["model_loaded"] is True

def test_predict_single_customer_success():
    """
    Test Case 2: Verify the /predict endpoint handles a perfectly structured
    incoming customer payload, returns HTTP 200, and builds the expected JSON schema.
    """
    valid_payload = {
        "customer_id": "CUST-4412",
        "total_spend": 189.50,
        "order_count": 4,
        "ticket_count": 1,
        "web_sessions": 22
    }
    response = client.post("/predict", json=valid_payload)

    # Assertions
    assert response.status_code == 200
    data = response.json()
    assert data["customer_id"] == "CUST-4412"
    assert "churn_probability" in data
    assert "predicted_class" in data
    assert "risk_explanation" in data
    assert isinstance(data["churn_probability"], float)
    assert data["predicted_class"] in [0, 1]

def test_predict_single_input_validation_failure():
    """
    Test Case 3: Verify that Pydantic input validation triggers an HTTP 422
    Unprocessable Entity error when bad data types or negative limits are supplied.
    """
    invalid_payload = {
        "customer_id": "CUST-FAULT",
        "total_spend": -25.00,  # CRITICAL: Triggers the ge=0.0 Pydantic boundary validation
        "order_count": "five",  # CRITICAL: Triggers a type mismatch (String instead of Int)
        "ticket_count": 0,
        "web_sessions": 10
    }
    response = client.post("/predict", json=invalid_payload)

    # Assertions
    assert response.status_code == 422
    errors = response.json()["detail"]

    # Ensure the error fields are explicitly caught by Pydantic's data validation layer
    error_fields = [err["loc"][-1] for err in errors]
    assert "total_spend" in error_fields
    assert "order_count" in error_fields